In [48]:
import sys
import os
from pathlib import Path
from datetime import timedelta
try:
    # Works in a standard .py script
    root = Path(__file__).resolve().parent.parent
except NameError:
    # Works in a Jupyter notebook (.ipynb)
    root = Path.cwd().parent 

# Add project root to sys.path to enable 'from src.data import ...'
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

# Now your imports will work
from src.data import (
    load_dataframe,
    save_dataframe,
    generate_missing_data_report,
    print_data_quality_summary
)

from src.data.calendar import create_gcc_holiday_calendar, align_to_trading_days

from src.utils.config import CONFIG, RAW_DATA_DIR, PROCESSED_DATA_DIR

def load_raw_data():
    """Load all raw data files."""
    print("\n" + "=" * 60)
    print("LOADING RAW DATA")
    print("=" * 60)

    universe = load_dataframe("gcc_banks_universe", RAW_DATA_DIR)
    fundamentals = load_dataframe("fundamentals_quarterly", RAW_DATA_DIR)
    market_data = load_dataframe("prices_daily", RAW_DATA_DIR)
    benchmarks = load_dataframe("benchmarks_daily", RAW_DATA_DIR)
    # fx_rates = load_dataframe("fx_rates_daily", RAW_DATA_DIR)  # DISABLED: Using LSEG USD conversion
    holidays_lseg = load_dataframe("holidays_calendar", RAW_DATA_DIR)

    return universe, fundamentals, market_data, benchmarks, holidays_lseg


def create_master_calendar(holidays_lseg):
    """Create comprehensive GCC holiday calendar."""
    print("\n" + "=" * 60)
    print("CREATING GCC MASTER CALENDAR")
    print("=" * 60)

    # Create calendar from 2010 to present
    holidays = create_gcc_holiday_calendar(
        start_date="2010-01-01",
        lseg_holidays=holidays_lseg
    )

    # Save
    metadata = {
        "description": "Comprehensive GCC holiday calendar",
        "sources": ["LSEG API (2016+)", "Hijri reconstruction", "National holidays"],
        "start_date": "2010-01-01"
    }
    save_dataframe(holidays, "gcc_holidays_full", PROCESSED_DATA_DIR, metadata)

    return holidays

In [49]:
universe, fundamentals, market_data, benchmarks, holidays_lseg = load_raw_data()
holidays = create_master_calendar(holidays_lseg)



LOADING RAW DATA
✓ Loaded gcc_banks_universe.parquet from /Users/milanszabo/PYTHON_PROJECTS/SRISK_GULF/data/raw
  Shape: (46, 3)
✓ Loaded fundamentals_quarterly.parquet from /Users/milanszabo/PYTHON_PROJECTS/SRISK_GULF/data/raw
  Shape: (390, 185)
✓ Loaded prices_daily.parquet from /Users/milanszabo/PYTHON_PROJECTS/SRISK_GULF/data/raw
  Shape: (4370, 138)
✓ Loaded benchmarks_daily.parquet from /Users/milanszabo/PYTHON_PROJECTS/SRISK_GULF/data/raw
  Shape: (4224, 7)
✓ Loaded holidays_calendar.parquet from /Users/milanszabo/PYTHON_PROJECTS/SRISK_GULF/data/raw
  Shape: (681, 5)

CREATING GCC MASTER CALENDAR
✓ Saved gcc_holidays_full.parquet to /Users/milanszabo/PYTHON_PROJECTS/SRISK_GULF/data/processed
  Shape: (1011, 4), Saved at: 2026-03-11T22:25:21.529008


In [50]:
aligned = align_and_merge_data(
            fundamentals=fundamentals,
            market_data=market_data,
            benchmarks=benchmarks,
            fx_rates=None,  # Not needed - data already in USD
            bank_universe=universe,
            convert_usd=False,  # Already converted by LSEG
            interpolate_prices=True,  # Fill small gaps (≤2 days) in prices
            max_interpolation_gap=2  # Maximum gap size to interpolate
        )

Forward-filling quarterly fundamentals to daily...
  Consolidating by taking first non-null value for each column per date...
Calculating benchmark returns...
Extracting prices and market caps...
Interpolating small gaps (≤ 2 days) in prices and market caps...
  ✓ Prices: filled 27165 gaps (42940 → 15775 missing)
  ✓ Market caps: filled 18810 gaps (32726 → 13916 missing)
Calculating bank returns...
Merging all data sources...
✓ Aligned data: 5059 days × 330 columns


/Users/milanszabo/PYTHON_PROJECTS/SRISK_GULF/.venv/lib/python3.11/site-packages/pandas/core/arrays/masked.py:691:RuntimeWarning: divide by zero encountered in log


In [60]:
mask = aligned.columns.str.contains('ENBD')
aligned.loc[:, mask]

,fund_ENBD.DU_Total Assets,fund_ENBD.DU_Common Equity - Total,fund_ENBD.DU_Total Liabilities,fund_ENBD.DU_Total Liabilities & Equity,mktcap_ENBD.DU,price_ENBD.DU,ret_ENBD.DU
Date,,,,,,,
2009-12-29,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2009-12-30,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2009-12-31,<NA>,<NA>,<NA>,<NA>,4464135119.45527,0.774423,<NA>
2010-01-03,<NA>,<NA>,<NA>,<NA>,4409651729.52099,0.764971,-0.01228
2010-01-04,<NA>,<NA>,<NA>,<NA>,4355168339.58671,0.75552,-0.012432
...,...,...,...,...,...,...,...
2026-03-04,317087928546.143005,36885058410.260597,277652424910.81897,317087928546.143005,53915259122.641502,8.53549,-0.013072
2026-03-05,317087928546.143005,36885058410.260597,277652424910.81897,317087928546.143005,51259361674.037399,8.115026,-0.050515
2026-03-06,317087928546.143005,36885058410.260597,277652424910.81897,317087928546.143005,50561678411.663803,8.004574,-0.013704
